In [1]:
import os
import json
import pandas as pd

from pathlib import Path


# Get GEO_ID from folder name (same pattern as prepare_data.ipynb)
GEO_ID = os.path.basename(os.path.dirname(os.path.abspath('__file__')))


description_path = Path('description.json')
preprocessed_csv_path = Path(f"{GEO_ID}_preprocessed.csv")

if not description_path.exists():
    raise FileNotFoundError(
        "description.json not found. Please run 'prepare_data.ipynb' first to generate it."
    )

if not preprocessed_csv_path.exists():
    raise FileNotFoundError(
        f"{preprocessed_csv_path} not found. Please run 'preprocess_data.ipynb' first to generate it."
    )

with open(description_path) as f:
    summary_data = json.load(f)

df = pd.read_csv(preprocessed_csv_path)

print("Data contains", df.shape[0], "samples and", df.shape[1], "columns.")
print("First few rows of the data:")
df.head()


Data contains 165 samples and 12676 columns.
First few rows of the data:


,month,cancer_death,sample_title,subtype,tnm,Stage,grading,cohort,serpinb3 expression (high_low),SAMD11,...,COX2,ATP8,ATP6,COX3,ND3,ND4L,ND4,ND5,ND6,CYTB
0,17.47,1,100_67161,classical,T3N0MX,IIA,G1,UMD-3,L,1.401006,...,6.185995,3.425500,5.478883,7.337665,4.451029,4.213376,5.934091,5.553229,3.183231,5.726467
1,25.33,1,106_62660,classical,T3N0MX,IIA,G2,UMD-3,H,1.688630,...,7.288531,4.186766,6.135954,8.346814,5.244528,5.013787,6.677582,5.962565,3.735310,5.767141
2,37.50,1,108_56008,classical,T2N0MX,IB,G2-3,UMD-3,H,1.407038,...,5.814629,3.156867,5.208641,7.012588,4.239895,4.165831,5.887478,5.654674,3.207215,5.438136
3,1.77,1,109_57484,unclassified,T3N1MX,IIB,G3,UMD-3,L,2.244019,...,6.285254,3.350332,5.579558,7.288593,4.861689,4.647450,6.397146,5.950132,3.310646,6.209400
4,14.20,1,11_19927T,basal,T2N0M0,IB,G3,UMD-1,H,1.930785,...,6.156480,3.227813,5.322206,7.222433,5.324452,4.160351,5.937567,5.280425,3.065631,5.681370


In [2]:
n_stats_written = 0

for idx, endpoint in enumerate(summary_data['survival-endpoints']):
    time_var = endpoint['time_var']['var_name']
    event_var = endpoint['event_var']['var_name']

    if time_var not in df.columns or event_var not in df.columns:
        abbrv = endpoint.get('abbrv', f'endpoint[{idx}]')
        print(
            f"Skipping stats for {abbrv}: columns not in preprocessed table "
            f"({time_var!r}, {event_var!r})."
        )
        endpoint.pop('stats', None)
        continue

    complete_mask = df[[time_var, event_var]].notnull().all(axis=1)
    n_complete = int(complete_mask.sum())
    n_incomplete = int(len(df) - n_complete)

    # Among samples with complete time + event data
    censored_count = int((complete_mask & (df[event_var] == 0.0)).sum())
    event_count = int((complete_mask & (df[event_var] == 1.0)).sum())

    censored_ratio = censored_count / n_complete

    print(f"Number of samples with incomplete data for {time_var} or {event_var}: {n_incomplete}")
    print(f"Number of samples with complete data for both variables: {n_complete}")
    print(f"Number of censored samples (event_var == 0.0): {censored_count}")
    print(f"Number of events (event_var == 1.0): {event_count}")
    pct = 100.0 * censored_ratio
    print(
        f"Censored ratio (among complete): {censored_count} / {n_complete} = "
        f"{censored_ratio:.3f} ({pct:.1f}%)"
    )

    # Refresh only this block on each run; leave all other endpoint fields unchanged
    endpoint['stats'] = {
        'n_incomplete': n_incomplete,
        'n_complete': n_complete,
        'n_censored': censored_count,
        'n_events': event_count,
        'censored_ratio': round(censored_ratio, 2),
    }
    n_stats_written += 1

with open(description_path, 'w', encoding='utf-8') as f:
    json.dump(summary_data, f, indent=4, ensure_ascii=False)
    f.write('\n')

print(
    f"Updated stats for {n_stats_written} of {len(summary_data['survival-endpoints'])} endpoint(s); "
    f"wrote {description_path.resolve()}"
)


Number of samples with incomplete data for month or cancer_death: 0
Number of samples with complete data for both variables: 165
Number of censored samples (event_var == 0.0): 50
Number of events (event_var == 1.0): 115
Censored ratio (among complete): 50 / 165 = 0.303 (30.3%)
Updated stats for 1 of 1 endpoint(s); wrote /Users/jakakokosar/dev/phd/survival-rnaseq-data/datasets/geo/GSE224564/description.json
